# Python程序设计（项目实战） · 工单P3《产线KPI仪表盘与自动日报》
**第5周 | 2课时(80分钟) | 理实一体**  | 学生实训版（TODO已补全，含运行结果）

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 数据钻取：点击折线点弹出该时段明细表（st.data_frame联动）；示范讲解 |
| **B 组长** | 标准达成 | 联动流畅+Plotly交互 |
| **C 操作员** | 基础完成 | 5KPI可用+时间筛选 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：Streamlit每次交互后脚本如何重跑？缓存用哪个装饰器？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 三、任务要求（Make）
**实施步骤**
1. 跑通starter最小仪表盘
2. 5个KPI: 趋势/良率/状态灯/告警/对标
3. 侧栏筛选(日期+设备)+cache_data

**验收标准**：及格(D)=`5KPI可用+时间筛选` ；优秀(B+)=`联动流畅+Plotly交互`

### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""P3 产线KPI仪表盘+自动日报 —— 参考实现(已补全)
运行: streamlit run W05_dashboard.py   # 仪表盘
      python W05_dashboard.py --report # 离线生成日报HTML
数据: 自动生成模拟产线数据(或替换为P2.1的clean.csv)
"""
import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

rng = np.random.RandomState(2026)

### Starter 2：`@st.cache_data`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
@st.cache_data
def load_data():
    """TODO-0(可选): 改为 pd.read_csv('clean.csv') 使用P2.1产出"""
    n = 2000
    ts = pd.date_range("2026-04-01", periods=n, freq="30min")
    output = rng.poisson(50, n)                      # 先抽 output 再抽 device_id（顺序固定）
    df = pd.DataFrame({
        "ts": ts,
        "device_id": rng.choice(["DEV-01", "DEV-02", "DEV-03", "DEV-04"], n),
        "output": output,
    })
    df["ok_qty"] = (df.output * rng.uniform(0.88, 0.99, n)).astype(int)
    df["yield"] = df.ok_qty / df.output
    df["temp"] = rng.uniform(45, 95, n)
    df["vib"] = rng.uniform(1, 6, n)
    return df

2026-10-12 18:47:07.016 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1 侧栏筛选: 日期范围(date_input) +
>   设备多选(multiselect)
> - TODO-2 四个KPI指标卡:
> - TODO-3 产量趋势: px.line(df, x='ts',
>   y='output') → st.plotly_chart TODO-4 良率仪
>   表/设备状态红绿灯(st.success/warning/error)/告警表格
>   (st.dataframe) TODO-5 效率对标: 进度条
>   st.progress(实际/目标)

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`5KPI可用+时间筛选`，优秀(B+)=`联动流畅+Plotly交互`

> **本工单第一大坑**：`yield` 是 Python 保留字，取列只能写 `df["yield"]`，
> 写 `df.yield` 会直接 `SyntaxError`（解析期报错，不是运行期）。

### Starter 3：`def main():`（TODO-1 ～ TODO-5 已补全）
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
def main():
    st.set_page_config("产线驾驶舱", "📊", layout="wide")
    st.title("🏭 产线KPI实时仪表盘")

    df = load_data()

    # ===== TODO-1 侧栏筛选: 日期范围(date_input) + 设备多选(multiselect) =====
    with st.sidebar:
        st.header("🔎 筛选条件")
        d_min, d_max = df.ts.dt.date.min(), df.ts.dt.date.max()
        picked = st.date_input("日期范围", value=(d_min, d_max),
                               min_value=d_min, max_value=d_max)
        devices = st.multiselect("设备", sorted(df.device_id.unique()),
                                 default=sorted(df.device_id.unique()))
    # date_input 返回 tuple(区间) 或 单日，两种情况都要兜住
    start = picked[0] if isinstance(picked, (list, tuple)) else picked
    end = picked[1] if isinstance(picked, (list, tuple)) and len(picked) > 1 else start
    df = df[(df.ts.dt.date >= start) & (df.ts.dt.date <= end)
            & df.device_id.isin(devices)]

    # ===== TODO-2 四个KPI指标卡 =====
    today = df[df.ts.dt.date == df.ts.dt.date.max()]
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("总产量", f"{int(df.output.sum()):,}")
    c2.metric("平均良率", f"{df['yield'].mean():.1%}")
    c3.metric("设备在线", f"{df.device_id.nunique()}/4")
    c4.metric("今日告警", f"{int((today['vib'] > 4.5).sum())} (振动>4.5)")

    # ===== TODO-3 产量趋势: px.line(df, x='ts', y='output') → st.plotly_chart =====
    fig = px.line(df, x="ts", y="output", title="产量趋势", render_mode="webgl")
    st.plotly_chart(fig, use_container_width=True)

    # ===== TODO-4 良率仪表/设备红绿灯(st.success/warning/error)/告警表格(st.dataframe) =====
    st.subheader("良率与设备状态")
    col_g, col_t = st.columns(2)
    with col_g:
        for dev in sorted(df.device_id.unique()):
            dev_df = df[df.device_id == dev]
            vib, temp = dev_df.iloc[-1].vib, dev_df.iloc[-1].temp
            if vib > 4.5 or temp > 90:
                st.error(f"{dev} 🔴 故障 | 振动{vib:.2f}mm/s 温度{temp:.1f}℃")
            elif vib > 3.5 or temp > 80:
                st.warning(f"{dev} 🟡 预警 | 振动{vib:.2f}mm/s 温度{temp:.1f}℃")
            else:
                st.success(f"{dev} 🟢 正常 | 振动{vib:.2f}mm/s 温度{temp:.1f}℃")
    with col_t:
        alarm = df[(df.vib > 4.5) | (df.temp > 90)].sort_values("ts", ascending=False)
        st.dataframe(alarm[["ts", "device_id", "vib", "temp", "output"]].head(20),
                     use_container_width=True)

    # ===== TODO-5 效率对标: 进度条 st.progress(实际/目标) =====
    target = 5000
    actual = int(today.output.sum())
    st.subheader("效率对标")
    st.progress(min(actual / target, 1.0))
    st.caption(f"今日 {actual:,} / 目标 {target:,} = {min(actual/target, 1.0):.1%}")

### Starter 4：`def build_report():`（TODO-1 ～ TODO-3 已补全）
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
def build_report():
    """P3.2 每日日报(四段式: 概览/分析/异常/附表) — python W05_dashboard.py --report 触发"""
    df = load_data()
    today = df[df.ts.dt.date == df.ts.dt.date.max()]

    # ---- TODO-1 班次/设备对比表 ----
    dev_tbl = (df.groupby("device_id")
                 .agg(产量=("output", "sum"), 平均良率=("yield", "mean"),
                      平均振动=("vib", "mean"), 平均温度=("temp", "mean"))
                 .round(3).to_html(classes="tbl"))
    shift = df.copy()
    shift["班次"] = shift.ts.dt.hour // 8
    shift["班次"] = shift["班次"].map({0: "白班(0-8h)", 1: "中班(8-16h)", 2: "夜班(16-24h)"})
    shift_tbl = (shift.groupby("班次")
                      .agg(产量=("output", "sum"), 平均良率=("yield", "mean"))
                      .round(3).to_html(classes="tbl"))

    # ---- TODO-2 自动列出 temp>90 或 vib>4.5 记录 + 建议 ----
    alarm = df[(df.temp > 90) | (df.vib > 4.5)].sort_values("ts", ascending=False)
    alarm_tbl = alarm[["ts", "device_id", "output", "yield", "temp", "vib"]].head(20).to_html(
        classes="tbl", index=False)
    advice = (f"<li>高振动 {int((df.vib > 4.5).sum())} 条 → 轴对中检查、动平衡校正</li>"
              f"<li>高温 {int((df.temp > 90).sum())} 条 → 检查冷却与润滑系统</li>"
              f"<li>良率<90% 共 {int((df['yield'] < 0.9).sum())} 条 → 关联工艺参数排查</li>")

    # ---- TODO-3 df.tail(50).to_html() ----
    tail_tbl = df.tail(50).to_html(classes="tbl", index=False)

    target = 5000
    actual = int(today.output.sum())
    rate = min(actual / target, 1.0)
    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>每日生产分析报告</title>
<style>
body {{ font-family: sans-serif; margin: 32px; line-height: 1.6; }}
h1 {{ color: #1a3c5e; border-bottom: 3px solid #2c7be5; padding-bottom: 8px; }}
h2 {{ color: #2c7be5; margin-top: 26px; }}
.tbl {{ width: 100%; border-collapse: collapse; margin: 12px 0; font-size: 14px; }}
.tbl th, .tbl td {{ border: 1px solid #dee2e6; padding: 8px; }}
.tbl th {{ background: #e9ecef; }}
.tbl tr:nth-child(even) {{ background: #f8f9fa; }}
.progress {{ width: 100%; height: 22px; background: #e9ecef; border-radius: 11px; overflow: hidden; }}
.fill {{ height: 100%; background: #28a745; width: {rate*100:.1f}%; }}
</style></head><body>
<h1>每日生产分析报告 {datetime.date.today()}</h1>
<h2>1. 生产概览</h2>
<p>总产量 <b>{int(df.output.sum()):,}</b> | 平均良率 <b>{df['yield'].mean():.1%}</b>
 | 设备在线 <b>{df.device_id.nunique()}/4</b>
 | 今日告警 <b>{int((today['vib'] > 4.5).sum())}</b>
 | 数据范围 {df.ts.dt.date.min()} ~ {df.ts.dt.date.max()} ({len(df):,} 条)</p>
<h2>2. 详细分析</h2>
<h3>2.1 设备对比</h3>{dev_tbl}
<h3>2.2 班次对比</h3>{shift_tbl}
<h2>3. 异常识别</h2>
<h3>3.1 告警明细（Top 20）</h3>{alarm_tbl}
<h3>3.2 处置建议</h3><ul>{advice}</ul>
<h2>4. 数据附表（最近50条）</h2>{tail_tbl}
<h2>5. 效率对标</h2>
<p>今日 {actual:,} / 目标 {target:,} = <b>{rate:.1%}</b></p>
<div class="progress"><div class="fill"></div></div>
<p>{"🎉 目标达成！" if rate >= 1 else f"⚠️ 还差 {target-actual:,} 件达标"}</p>
<hr><p style="color:#6c757d;font-size:.9em">生成时间: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}
 | 数据来源: 模拟产线数据 (RandomState=2026) | reportlab转PDF复用02包w10字体方案</p>
</body></html>"""
    with open("daily_report.html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ 日报已生成 daily_report.html | 总产量 {int(df.output.sum()):,} "
          f"| 平均良率 {df['yield'].mean():.1%} | 效率对标 {rate:.1%}"
          "  (reportlab转PDF, 复用02包w10字体方案)")

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
if __name__ == "__main__":
    import sys
    # 注意：Notebook 内核中 __name__ 同样是 "__main__"，但 st.* 无法在浏览器外渲染，
    # 直接调 main() 只会刷 Streamlit bare 模式的 WARNING（不报错但很吵）。
    # 命令行里才是本来的分流：--report → 生成日报；否则起 Streamlit 仪表盘。
    if "--report" in sys.argv:
        build_report()
    elif "ipykernel" in sys.modules:
        print("Notebook 环境：跳过 Streamlit 仪表盘（st.* 在此无法渲染）")
        print("  仪表盘请用命令行: streamlit run W05_dashboard.py")
        print("  日报请用命令行  : python W05_dashboard.py --report")
        print("  在 Notebook 内验证日报：直接调用下方第四节的 build_report()")
    else:
        main()

Notebook 环境：跳过 Streamlit 仪表盘（st.* 在此无法渲染）
  仪表盘请用命令行: streamlit run W05_dashboard.py
  日报请用命令行  : python W05_dashboard.py --report
  在 Notebook 内验证日报：直接调用下方第四节的 build_report()


## 四、Notebook内直接验证（Streamlit外跑通数据层+日报）

> Streamlit的`main()`需要在浏览器中运行（`streamlit run W05_dashboard.py`）。
> 在Notebook里直接调用同一个 `build_report()`（不依赖浏览器），
> 验证**同一份数据层与同一套日报逻辑**能产出完整四段式日报，保证"程序出结果"（D层验收）。
> `@st.cache_data` 在非 Streamlit 环境会自动降级为内存缓存，仅有 WARNING，可正常调用。

### 1) 直接调用本 notebook 已补全的 `build_report()` 生成日报

In [7]:
import os

build_report()          # 调用上方 Starter 4 已补全的函数

print("\n--- daily_report.html 大小: %d bytes ---" % os.path.getsize("daily_report.html"))
with open("daily_report.html", encoding="utf-8") as f:
    print(f.read()[:480])

✅ 日报已生成 daily_report.html | 总产量 100,414 | 平均良率 92.5% | 效率对标 32.5%  (reportlab转PDF, 复用02包w10字体方案)

--- daily_report.html 大小: 16234 bytes ---
<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>每日生产分析报告</title>
<style>
body { font-family: sans-serif; margin: 32px; line-height: 1.6; }
h1 { color: #1a3c5e; border-bottom: 3px solid #2c7be5; padding-bottom: 8px; }
h2 { color: #2c7be5; margin-top: 26px; }
.tbl { width: 100%; border-collapse: collapse; margin: 12px 0; font-size: 14px; }
.tbl th, .tbl td { border: 1px solid #dee2e6; padding: 8px; }
.tbl th { background: #e9ecef; }
.tbl tr:nth-child(even) { background


### 2) KPI五件套核验（Notebook内直算，与日报同一数据层）

In [8]:
df = load_data()
today = df[df.ts.dt.date == df.ts.dt.date.max()]

print("数据范围:", df.ts.dt.date.min(), "~", df.ts.dt.date.max(), f"({len(df):,} 条)\n")
print("--- KPI五件套 ---")
print(f"KPI-1 总产量   : {int(df.output.sum()):,}")
print(f"KPI-2 平均良率 : {df['yield'].mean():.1%}")
print(f"KPI-3 设备在线 : {df.device_id.nunique()}/4")
print(f"KPI-4 今日告警 : {int((today['vib'] > 4.5).sum())} (振动>4.5mm/s)")
print(f"KPI-5 效率对标 : {min(int(today.output.sum())/5000, 1.0):.1%}"
      f" ({int(today.output.sum()):,}/5,000)")

# 产量趋势（日度）
daily = df.groupby(df.ts.dt.date).agg(产量=("output", "sum"), 良率=("yield", "mean"))
print("\n--- 产量/良率趋势（日度, 前3天）---")
print(daily.head(3).round(3).to_string())

# 设备状态红绿灯
print("\n--- 设备状态红绿灯 ---")
for dev in sorted(df.device_id.unique()):
    d = df[df.device_id == dev]
    vib, temp = d.iloc[-1].vib, d.iloc[-1].temp
    level = "🔴故障" if (vib > 4.5 or temp > 90) else \
            ("🟡预警" if (vib > 3.5 or temp > 80) else "🟢正常")
    print(f"  {dev}: {level} (振动{vib:.2f}mm/s, 温度{temp:.1f}℃)")

print("\n--- 告警统计 ---")
print(f"  全期振动>4.5 : {int((df.vib > 4.5).sum())} 条")
print(f"  全期温度>90  : {int((df.temp > 90).sum())} 条")
print(f"  合计异常记录 : {int(((df.vib > 4.5) | (df.temp > 90)).sum())} 条")

assert df.device_id.nunique() == 4
assert 0 < df["yield"].mean() < 1
print("\n5个KPI全部可用 → 达到及格(D)标准 `5KPI可用+时间筛选`")

数据范围: 2026-04-01 ~ 2026-05-12 (2,000 条)

--- KPI五件套 ---
KPI-1 总产量   : 100,414
KPI-2 平均良率 : 92.5%
KPI-3 设备在线 : 4/4
KPI-4 今日告警 : 9 (振动>4.5mm/s)
KPI-5 效率对标 : 32.5% (1,625/5,000)

--- 产量/良率趋势（日度, 前3天）---
              产量     良率
ts                     
2026-04-01  2435  0.924
2026-04-02  2437  0.923
2026-04-03  2404  0.928

--- 设备状态红绿灯 ---
  DEV-01: 🟡预警 (振动3.77mm/s, 温度77.9℃)
  DEV-02: 🔴故障 (振动5.48mm/s, 温度80.8℃)
  DEV-03: 🟢正常 (振动2.14mm/s, 温度49.3℃)
  DEV-04: 🔴故障 (振动5.60mm/s, 温度70.7℃)

--- 告警统计 ---
  全期振动>4.5 : 610 条
  全期温度>90  : 194 条
  合计异常记录 : 755 条

5个KPI全部可用 → 达到及格(D)标准 `5KPI可用+时间筛选`


### 3) 多解互验（数据口径一致性）

> `教师参考/solutions_multi.py` 里的 `W05_a()` 用的是 **n=500 + random.Random(2026).gauss**，
> 与本工单 **n=2000 + RandomState(2026).poisson** 是两套不同数据源，数值本就不相等；
> 该库内部已验证 `W05_a() == W05_b()`（解A=解B，22项互验 ALL PASS）。
> 本工单的"两解"=**解A(Streamlit实时数据层)/解B(静态HTML日报)**，差异只在呈现层，
> 所以正确的一致性验证是：**同一数据层、两种呈现，关键指标必须相等**。

In [9]:
import importlib.util

spec = importlib.util.spec_from_file_location("solutions_multi", "教师参考/solutions_multi.py")
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)

print("--- 教师参考库 solutions_multi.py 加载验证 ---")
print("  W05_a() 返回:", m.W05_a())
print("  W05_a() == W05_b() (解A=解B):", m.W05_a() == m.W05_b())

html = open("daily_report.html", encoding="utf-8").read()
print("\n--- 本工单两解口径一致验证（数据层直算 vs HTML日报）---")
print("  解A Streamlit数据层: 总产量 %s | 平均良率 %.1f%% | 设备在线 %d/4 | 效率对标 %.1f%%"
      % (f"{int(df.output.sum()):,}", df["yield"].mean() * 100,
         df.device_id.nunique(),
         min(int(today.output.sum()) / 5000, 1.0) * 100))
assert f"{int(df.output.sum()):,}" in html, "日报总产量与数据层不一致"
assert f"{df['yield'].mean():.1%}" in html, "日报良率与数据层不一致"
assert f"{df.device_id.nunique()}/4" in html, "日报设备在线数与数据层不一致"
print("  解B 静态HTML日报  : 总产量/平均良率/设备在线 三项与解A完全一致")
print("\n两解数据口径一致 → 课堂‘车间大屏选A、管理层日报选B’对比结论可信")

--- 教师参考库 solutions_multi.py 加载验证 ---
  W05_a() 返回: {datetime.date(2026, 4, 9): 2296, datetime.date(2026, 4, 10): 2397, datetime.date(2026, 4, 11): 978}
  W05_a() == W05_b() (解A=解B): True

--- 本工单两解口径一致验证（数据层直算 vs HTML日报）---
  解A Streamlit数据层: 总产量 100,414 | 平均良率 92.5% | 设备在线 4/4 | 效率对标 32.5%
  解B 静态HTML日报  : 总产量/平均良率/设备在线 三项与解A完全一致

两解数据口径一致 → 课堂‘车间大屏选A、管理层日报选B’对比结论可信


## 四、提交要求与避坑
- 提交: `学号_姓名_P3.py` 或本notebook(含输出)
- ⚠ Plotly图记得st.plotly_chart(py.plotly_fig)渲染
- ⚠ 时间筛选用st.date_input的区间值（返回tuple，要判断长度）
- ⚠ 缓存键=数据文件mtime，自动失效
- ⚠ `df.yield` 是语法错误（yield是保留字），必须写 `df["yield"]`